In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/customer_support_tickets.csv
/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/enhanced_customer_support_data.csv


In [2]:
!pip install --upgrade transformers accelerate datasets evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 100.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 114.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.5 MB/s eta 0:00:0000:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully un

In [9]:
import os
import pandas as pd
import numpy as np
import torch
import gc
import logging
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import logging as hf_logging
from tqdm.auto import tqdm

# Suppress the stubborn warnings
hf_logging.set_verbosity_error()

# 1. LOAD DATA
file_path = '/kaggle/input/datasets/ajverse/customer-support-tickets-crm-dataset/customer_support_tickets.csv'
df = pd.read_csv(file_path).copy()

# 2. LOAD MODEL (NATIVE - NO REMOTE CODE BUG)
print("Loading Native Phi-3-mini...")
model_id = "microsoft/Phi-3-mini-4k-instruct"

# Notice we removed trust_remote_code=True! We are using stable, native code now.
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token 

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto", 
    torch_dtype=torch.float16
)
print("Phi-3 Successfully Loaded!")

# 3. PREPARE ALL PROMPTS
print("Preparing messages for GPU Batching...")
all_prompts = []
for text in df['Ticket_Description'].fillna(""):
    messages = [
        {"role": "system", "content": "You are a strict data auditor. Read the support ticket and reply with EXACTLY ONE NUMBER indicating severity: 1 (Low), 2 (Medium), 3 (High), or 4 (Critical). Do not provide any other text."},
        {"role": "user", "content": f"Ticket: '{text}'"}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    all_prompts.append(prompt)

# 4. RUN NATIVE BATCH INFERENCE (DIRECT PYTORCH)
print("Running High-Speed LLM Inference via PyTorch...")
all_llm_scores = []
batch_size = 16

for i in tqdm(range(0, len(all_prompts), batch_size), desc="Processing Batches"):
    
    batch_prompts = all_prompts[i : i + batch_size]
    
    inputs = tokenizer(
        batch_prompts, return_tensors="pt", padding=True, truncation=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,        
            do_sample=False,         
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True           # High-speed memory is back on and bug-free!
        )
    
    prompt_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[:, prompt_length:]
    responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    for j, response in enumerate(responses):
        response = response.strip()
        
        if i == 0 and j == 0:
            print(f"\n[DEBUG] First Ticket Raw Output: '{response}'\n")
        
        score = 2 # Default fallback
        for num in ['4', '3', '2', '1']:
            if num in response: 
                score = int(num)
                break
        all_llm_scores.append(score)

df['score_llm'] = all_llm_scores

# 5. FUSION PIPELINE & ABLATION STUDY
print("\nApplying logic rules and fusing scores...")

def signal_resolution_time(hours_to_resolve):
    if pd.isna(hours_to_resolve): return 2 
    try: hours = float(hours_to_resolve)
    except ValueError: return 2
    if hours > 120: return 4    
    elif hours > 48: return 3   
    elif hours > 12: return 2   
    else: return 1  

df['score_time'] = df['Resolution_Time_Hours'].apply(signal_resolution_time)
df['fused_score'] = (df['score_llm'] * 0.7) + (df['score_time'] * 0.3)
df['inferred_severity_num'] = df['fused_score'].round().astype(int)

severity_map = {1: 'Low', 2: 'Medium', 3: 'High', 4: 'Critical'}
df['inferred_severity'] = df['inferred_severity_num'].map(severity_map)

# --- ABLATION METRICS ---
print("\n--- ABLATION STUDY METRICS ---")
df['llm_only_severity'] = df['score_llm'].map(severity_map)
df['time_only_severity'] = df['score_time'].map(severity_map)

print(f"LLM-Only Match Rate:   {(df['Priority_Level'] == df['llm_only_severity']).mean():.2%}")
print(f"Time-Only Match Rate:  {(df['Priority_Level'] == df['time_only_severity']).mean():.2%}")
print(f"Fused Pipeline Match:  {(df['Priority_Level'] == df['inferred_severity']).mean():.2%}")

# Derive the final mismatch label required for Stage 2
df['Mismatch'] = np.where(df['Priority_Level'] != df['inferred_severity'], 1, 0)

output_path = '/kaggle/working/processed_tickets_final.csv'
df.to_csv(output_path, index=False)
print(f"\nStage 1 Complete! Saved to {output_path}")

# 6. FREE UP GPU MEMORY
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

Loading Native Phi-3-mini...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Phi-3 Successfully Loaded!
Preparing messages for GPU Batching...
Running High-Speed LLM Inference via PyTorch...


Processing Batches:   0%|          | 0/1250 [00:00<?, ?it/s]


[DEBUG] First Ticket Raw Output: '1 (Low'


Applying logic rules and fusing scores...

--- ABLATION STUDY METRICS ---
LLM-Only Match Rate:   30.35%
Time-Only Match Rate:  26.25%
Fused Pipeline Match:  29.45%

Stage 1 Complete! Saved to /kaggle/working/processed_tickets_final.csv
